# MiniMax H3 Colab Setup

Before running the notebook:

1. **Strongly recommended: select an A100 high-RAM GPU.** In Colab, choose **Runtime > Change runtime type > A100 high-RAM**. This notebook has very high memory requirements; an A100 with 80 GB VRAM and at least 60 GB system RAM provides the most reliable experience.
2. **Create your Hugging Face token:** Open [Hugging Face Settings > Access Tokens](https://huggingface.co/settings/tokens), sign in, select **Create new token**, choose the **Read** permission, and copy the token. In Colab, open the **Secrets** tab (key icon) in the left sidebar, create a secret named `HF_TOKEN`, paste the token as its value, and enable **Notebook access**. Never commit or share the token. The model-loading cell reads this secret. If it is unavailable, Colab will ask for the token privately.
3. **Run the code cells in order:**
   - The first code cell installs SageAttention, mounts Google Drive, and caches its wheel at `/content/drive/MyDrive/wheels/sageattention`. The first run builds the wheel; later runs reuse it.
   - The next code cell installs the remaining packages and configures the environment. Set `USE_DRIVE = True` there if you also want model files persisted at `DRIVE_CACHE`.
   - Run the restart cell containing `os._exit(00)`. The runtime will intentionally disconnect or appear to crash; this is expected behavior. Wait for Colab to reconnect, then manually run the model-loading cell followed by the generation GUI cell.
4. **Turbo LoRA:** To enable the few-step adapter, set `FUSE_TURBO_LORA = True` (and optionally `TURBO_LORA_STRENGTH`, 1.0 is what it was tuned for) in the model-loading cell before running it. The adapter downloads (~744 MB) and is merged into the transformer's INT8 weights as the model loads, so peak VRAM during generation matches the base model and long clips still fit. With it on, use 4-8 steps. It costs a little sharpness (a second quantization rounding), applies to T2V and I2V only, and lasts for the session — re-run the loading cell with the flag off to get an unmodified transformer back.
5. **Memory mode:** The model-loading cell measures VRAM and system RAM automatically. It uses a fast GPU-resident mode on high-memory hardware and switches to slower CPU-RAM group offloading when needed. Disk offload is disabled because TorchAO quantized tensors cannot use Diffusers disk offload. Google Drive is used only for persistence. Set `FORCE_LOW_MEMORY_MODE` in that cell only if you need to override automatic selection.

The model download is large, so keep the Drive mount enabled if you want to avoid downloading it again in a new runtime.

In [ ]:
!pip install -U torchao av -q
!pip install kernels -q

# ╔══════════════════════════════════════════════════════════════╗
# ║  ENSURE SAGEATTENTION — install cached wheel or build+cache   ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, torch, glob, os

# ── Mount Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

WHEEL_DIR = "/content/drive/MyDrive/wheels/sageattention"
os.makedirs(WHEEL_DIR, exist_ok=True)

def sh(cmd):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True).returncode

# ── Step 1: try any cached wheel already on Drive ──────────────
# Env fingerprint lives in the wheel's build-tag segment (see below),
# so this stays a flat directory — just grab whatever's newest.
cached_wheels = sorted(glob.glob(f"{WHEEL_DIR}/*.whl"))
installed_ok = False

if cached_wheels:
    latest = cached_wheels[-1]
    print(f"Found cached wheel: {latest}")
    sh("pip uninstall -y sageattention")
    rc = sh(f"pip install --no-deps '{latest}'")
    installed_ok = (rc == 0)
    if not installed_ok:
        print("✗ Cached wheel failed to install — will rebuild from source.")
else:
    print("No cached wheel found on Drive.")

# ── Step 2: build from source if cache missing/broken ──────────
if not installed_ok:
    print("\nBuilding sageattention from source for this exact environment …")
    sh("pip uninstall -y sageattention")

    cap_major, cap_minor = torch.cuda.get_device_capability()
    arch = f"{cap_major}.{cap_minor}"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Detected GPU: {gpu_name}  (sm_{cap_major}{cap_minor})")

    build_dir = "/content/sage_wheel_build"
    sh(f"rm -rf {build_dir} && mkdir -p {build_dir}")

    rc = sh(
        f"TORCH_CUDA_ARCH_LIST='{arch}' pip wheel --no-build-isolation --no-deps "
        f"git+https://github.com/thu-ml/SageAttention.git -w {build_dir}"
    )
    if rc != 0:
        raise RuntimeError("Source build failed — check the pip wheel output above.")

    built = glob.glob(f"{build_dir}/*.whl")
    if not built:
        raise RuntimeError("Build reported success but no .whl was produced.")
    built_wheel = built[0]

    rc = sh(f"pip install --no-deps '{built_wheel}'")
    if rc != 0:
        raise RuntimeError("Freshly built wheel failed to install.")

    print("✓ Freshly built wheel installed.")

    # Embed env fingerprint as a wheel BUILD TAG, not a filename prefix or
    # subdirectory. Per the wheel spec, filenames are:
    #   {distribution}-{version}(-{build_tag})?-{python_tag}-{abi_tag}-{platform_tag}.whl
    # pip only validates distribution-version against .dist-info, and the
    # build tag must start with a digit — so inserting one here is safe and
    # keeps everything as a single flat file (no subdirectories needed).
    build_tag = f"0cu{torch.version.cuda.replace('.', '')}sm{cap_major}{cap_minor}"

    base = os.path.basename(built_wheel)                     # e.g. sageattention-2.2.0-cp312-cp312-linux_x86_64.whl
    parts = base[:-4].split("-")                              # strip .whl, split on '-'
    dist, ver, py_tag, abi_tag, plat_tag = parts[0], parts[1], parts[2], parts[3], parts[4]
    tagged_name = f"{dist}-{ver}-{build_tag}-{py_tag}-{abi_tag}-{plat_tag}.whl"

    dest = f"{WHEEL_DIR}/{tagged_name}"
    sh(f"cp '{built_wheel}' '{dest}'")
    print(f"Cached to Drive: {dest}")
    installed_ok = True

# ── Step 3: reminder ─────────────────────────────────────────
print("\n" + "="*60)
print("IMPORTANT: restart the Colab runtime now before importing")
print("diffusers, so its package-detection cache picks this up.")
print("="*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install & Configure                               ║
# ║  Safe to re-run; pip skips already-installed packages.      ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os, json
import psutil

print("=" * 58)
print("  MiniMax H3 · Diffusers · A100 80 GB")
print("=" * 58)

# ── System check ─────────────────────────────────────────────
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print(f"\n✓ GPU        : {smi}")
    if "A100" not in smi:
        print("  ⚠  Not an A100 — timings in the GUI may be off")
except Exception as e:
    print(f"  GPU check failed: {e}")

vm = psutil.virtual_memory()
print(f"✓ System RAM : {vm.total/1e9:.0f} GB total, {vm.available/1e9:.0f} GB free")
if vm.total / 1e9 < 60:
    print("  ⚠  Need ≥ 75 GB RAM — Qwen3-VL-32B text encoder offloads to CPU RAM")

disk = psutil.disk_usage("/content")
print(f"✓ Disk       : {disk.free/1e9:.0f} GB free / {disk.total/1e9:.0f} GB")
if disk.free / 1e9 < 20:
    print("  ⚠  Low disk — model downloads to Drive, outputs go to /content")

# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURE HERE                                             ║
# ╚══════════════════════════════════════════════════════════════╝

# Mount Google Drive to persist the ~62 GB model download.
# Set False to skip (model re-downloads every new runtime).
USE_DRIVE = False

# Folder on Drive where HuggingFace cache is stored.
DRIVE_CACHE = "/content/drive/MyDrive/hf_cache_minimax_h3"

# ╚══════════════════════════════════════════════════════════════╝

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        os.makedirs(DRIVE_CACHE, exist_ok=True)
        # Redirect HF download cache to Drive before any imports
        os.environ["HF_HOME"] = DRIVE_CACHE
        print(f"✓ Drive      : HF cache → {DRIVE_CACHE}")
    except Exception as e:
        print(f"  Drive mount skipped ({e}) — using ephemeral /root/.cache")
else:
    print("  Drive skipped — weights won't persist across sessions")

os.makedirs("/content/h3_outputs", exist_ok=True)

# ── Packages ─────────────────────────────────────────────────
print("\nInstalling packages (skips if already installed) …")

def pip(*pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + list(pkgs),
        check=True,
    )

# Diffusers from source — H3 support is in main, not yet in a stable release
pip("git+https://github.com/huggingface/diffusers.git")

# torchao — INT8 / FP8 quantisation
pip("torchao")

# peft — required backend for load_lora_weights / set_adapters
pip("peft")

# Supporting stack
pip("accelerate", "transformers", "sentencepiece", "tokenizers")

# ipywidgets — GUI in notebook cells
pip("ipywidgets")

# Pillow — image handling
pip("pillow")

# av (PyAV) — video/audio decoding for I2V reference frames
pip("av")

# ffmpeg — needed by encode_video to mux audio into MP4
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=True)
print("✓ ffmpeg ready")

print("\n✓ Cell 1 complete — run Cell 2 to load the model")

In [ ]:
import os
print("Restarting Runtime...Crash is expected")
os._exit(00)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load Pipeline (INT8, run once per session)        ║
# ║                                                             ║
# ║  First run: downloads ~62 GB — allow 15–20 min             ║
# ║  Cached runs: loads from Drive in ~3–5 min                 ║
# ║                                                             ║
# ║  INT8 strategy (per official Diffusers docs):               ║
# ║    transformer  BF16→INT8  61.7 GB → ~31 GB                ║
# ║    text encoder BF16→INT8  62.1 GB → ~31 GB                ║
# ║    combined ~62 GB sits within A100 80 GB — no full-model   ║
# ║    CPU swaps during denoising, only block-level streaming   ║
# ║                                                             ║
# ║  Memory mode is selected automatically from VRAM and RAM.   ║
# ║  Low-memory mode uses CPU group offloading only.             ║
# ╚══════════════════════════════════════════════════════════════╝
import os

# Read once, when the CUDA caching allocator initialises — hence above the
# torch import. Merging a LoRA into resident INT8 weights rewrites every
# targeted Linear in place, which riddles a fixed-segment pool with holes;
# expandable segments let those holes be reused for the large activation
# tensors long clips need.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import psutil
import torch
from diffusers import (
    MiniMaxH3Transformer3DModel,
    ModularPipeline,
    TorchAoConfig,
)
from diffusers.hooks import apply_group_offloading
from transformers import Qwen3VLForConditionalGeneration
from transformers import TorchAoConfig as TransformersTorchAoConfig
from torchao.quantization import Int8WeightOnlyConfig

REPO = "MiniMaxAI/MiniMax-H3"
from huggingface_hub import login

# ── Memory preflight ───────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select an A100 high-RAM Colab runtime.")

gpu_properties = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu_properties.total_memory / 2**30
gpu_free_gib = torch.cuda.mem_get_info()[0] / 2**30
ram = psutil.virtual_memory()
sys_total_gib = ram.total / 2**30
sys_available_gib = ram.available / 2**30

FAST_MIN_VRAM_GIB = 70.0
FAST_MIN_SYS_RAM_GIB = 75.0
FAST_MIN_AVAILABLE_RAM_GIB = 64.0
MIN_LOAD_VRAM_GIB = 34.0
MIN_LOAD_AVAILABLE_RAM_GIB = 36.0
CPU_OFFLOAD_MIN_RAM_GIB = 70.0
FORCE_LOW_MEMORY_MODE = None  # None = automatic, True/False = override

# ── Turbo LoRA ────────────────────────────────────────────────
# larryvrh's few-step adapter for the fl2va partition, merged into the
# transformer's weights at load time — before any offload hook exists and
# while the allocator is still nearly empty, which is where merging costs
# the least fragmentation.
#
# Cost: dequantize -> add delta -> requantize means a second rounding, so
# output softens slightly. With it on, use 4-8 steps.
FUSE_TURBO_LORA     = False   # True = merge the adapter as the model loads
TURBO_LORA_STRENGTH = 1.0     # baked in at merge time; 1.0 is what it was tuned for

LORA_REPO         = "larryvrh/MiniMax-H3-Turbo-Lora"
LORA_WEIGHT       = "minimax_h3_turbo_v4_step600_ema.safetensors"
LORA_ADAPTER_NAME = "minimax_h3_turbo_v4"

hardware_needs_low_memory = (
    gpu_total_gib < FAST_MIN_VRAM_GIB
    or sys_total_gib < FAST_MIN_SYS_RAM_GIB
    or sys_available_gib < FAST_MIN_AVAILABLE_RAM_GIB
)
LOW_MEMORY_MODE = (
    hardware_needs_low_memory
    if FORCE_LOW_MEMORY_MODE is None
    else bool(FORCE_LOW_MEMORY_MODE)
)

local_disk = psutil.disk_usage("/content")
local_disk_gib = local_disk.free / 2**30
print(f"✓ GPU memory : {gpu_free_gib:.1f} GiB free / {gpu_total_gib:.1f} GiB total")
print(f"✓ System RAM : {sys_available_gib:.1f} GiB available / {sys_total_gib:.1f} GiB total")
print(f"✓ Local disk : {local_disk_gib:.1f} GiB free / {local_disk.total / 2**30:.1f} GiB total")
print(f"✓ Memory mode: {'LOW' if LOW_MEMORY_MODE else 'FAST'}")

if gpu_total_gib < MIN_LOAD_VRAM_GIB or gpu_free_gib < MIN_LOAD_VRAM_GIB:
    raise RuntimeError(
        f"Only {gpu_free_gib:.1f} GiB VRAM is available. "
        f"At least {MIN_LOAD_VRAM_GIB:.0f} GiB is needed even in low-memory mode. "
        "Select an A100 high-RAM runtime and close other GPU workloads."
    )
if sys_available_gib < MIN_LOAD_AVAILABLE_RAM_GIB:
    raise RuntimeError(
        f"Only {sys_available_gib:.1f} GiB system RAM is available. "
        f"At least {MIN_LOAD_AVAILABLE_RAM_GIB:.0f} GiB is needed to quantize one "
        "component safely. Select the A100 high-RAM runtime or restart the runtime."
    )
if LOW_MEMORY_MODE and sys_available_gib < CPU_OFFLOAD_MIN_RAM_GIB:
    raise RuntimeError(
        f"Only {sys_available_gib:.1f} GiB system RAM is available for CPU offload. "
        f"At least {CPU_OFFLOAD_MIN_RAM_GIB:.0f} GiB is recommended in low-memory mode. "
        "Free RAM and run this cell again. Google Drive and disk offload are not used."
    )

# TorchAO quantized tensors cannot be serialized for Diffusers disk offload.
# All low-memory paths therefore keep offloaded weights in CPU memory.
print(
    f"✓ Offload mode: "
    f"{'CPU' if LOW_MEMORY_MODE else 'GPU resident'}"
)

_offload = dict(
    onload_device=torch.device("cuda"),
    offload_device=torch.device("cpu"),
    use_stream=not LOW_MEMORY_MODE,
)
if LOW_MEMORY_MODE:
    _offload["low_cpu_mem_usage"] = True

# Keep this helper name because it is also used by the lazy Ref2VA loader.
# It intentionally never sets a disk path.
def memory_offload_kwargs():
    return dict(_offload)

# ── Auth ──────────────────────────────────────────────────────
# Preferred: Colab Secrets (key icon, left sidebar) → add a secret
# named HF_TOKEN → grant this notebook access. Falls back to a hidden
# prompt if no secret is found, so nothing is ever stored in the file.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HuggingFace token (input hidden, not saved): ")

login(token=HF_TOKEN.strip())

# ── INT8 — modules kept at BF16 ──────────────────────────────
# Input/output projections and normalisation layers are left at
# BF16: they are small and sensitive to quantisation error.
TRANSFORMER_SKIP = [
    "proj_in", "audio_proj_in", "context_embedder",
    "time_embedder", "time_proj", "token_refiner",
    "norm_out", "proj_out", "audio_proj_out",
]
ENCODER_SKIP = [
    "model.visual",
    "model.language_model.embed_tokens",
    "model.language_model.norm",
    "lm_head",
]

# ── Load transformer in INT8 ──────────────────────────────────
# device_map="cuda" quantizes layer-by-layer on GPU instead of
# staging the full ~62GB BF16 model in system RAM first.
print("Loading transformer (INT8) …  ~31 GB when quantised")
transformer = MiniMaxH3Transformer3DModel.from_pretrained(
    REPO,
    subfolder="transformer",
    dtype=torch.bfloat16,
    quantization_config=TorchAoConfig(
        Int8WeightOnlyConfig(version=2),   # version=2 tensors are pinnable
        modules_to_not_convert=TRANSFORMER_SKIP,
    ),
    low_cpu_mem_usage=True,       # required whenever quantization_config is set
    device_map="cuda",            # quantize on GPU, not host RAM
)
transformer.requires_grad_(False)          # freeze — no autograd through INT8
transformer = transformer.to("cpu")        # move quantized copy to CPU memory
# Group offloading is applied further down, after the optional LoRA merge:
# peft must inject into resident modules, and the merge must not fight
# onload/offload hooks.
torch.cuda.empty_cache()
print("  ✓ transformer loaded")

# ── Load text encoder (Qwen3-VL-32B) in INT8 ─────────────────
print("Loading text encoder (INT8) …  ~31 GB when quantised")
text_encoder = Qwen3VLForConditionalGeneration.from_pretrained(
    REPO,
    subfolder="text_encoder",
    dtype=torch.bfloat16,
    quantization_config=TransformersTorchAoConfig(
        Int8WeightOnlyConfig(version=2),
        modules_to_not_convert=ENCODER_SKIP,
    ),
    low_cpu_mem_usage=True,
    device_map="cuda",
)
text_encoder.requires_grad_(False)
text_encoder = text_encoder.to("cpu")
if LOW_MEMORY_MODE:
    apply_group_offloading(
        text_encoder.model,
        offload_type="leaf_level",
        **memory_offload_kwargs(),
    )
torch.cuda.empty_cache()
print("  ✓ text encoder loaded")

# ── Assemble pipeline ─────────────────────────────────────────
print("Assembling pipeline and loading VAEs …")
pipe = ModularPipeline.from_pretrained(REPO)
pipe.update_components(
    transformer=transformer,
    text_encoder=text_encoder,
)
# fl2va covers both T2V (no image arg) and I2V (image= arg)
pipe.load_components(workflow="fl2va", dtype=torch.bfloat16)

pipe.transformer.set_attention_backend("_sage_qk_int8_pv_fp16_cuda")


# ── Optional: merge the Turbo LoRA ────────────────────────────
def fuse_turbo_lora(strength: float):
    """Merge the adapter into the INT8 weights and drop the peft wrappers."""
    from torchao.utils import TorchAOBaseTensor

    tf = pipe.transformer

    def quantized_params():
        return sum(isinstance(p.data, TorchAOBaseTensor) for p in tf.parameters())

    before = quantized_params()

    print(f"Merging Turbo LoRA at strength {strength:.2f} …")
    pipe.load_lora_weights(
        LORA_REPO,
        weight_name=LORA_WEIGHT,
        adapter_name=LORA_ADAPTER_NAME,
        low_cpu_mem_usage=True,
    )
    # `get_delta_weight` multiplies by the adapter scaling, so this is what
    # bakes `strength` in; `fuse_lora` below keeps lora_scale at 1.0.
    tf.set_adapters([LORA_ADAPTER_NAME], weights=[float(strength)])
    tf.enable_lora()

    # peft's TorchaoLoraLinear.merge() needs a callback to requantize after
    # writing the merged weight. Transformers supplies it via its quantizer;
    # diffusers does not, so hand it over here. Modules in TRANSFORMER_SKIP
    # are still bf16 and merge natively.
    try:
        from peft.tuners.lora.torchao import TorchaoLoraLinear
    except ImportError:
        TorchaoLoraLinear = None
    if TorchaoLoraLinear is not None:
        for module in tf.modules():
            if isinstance(module, TorchaoLoraLinear):
                module.get_apply_tensor_subclass = (
                    lambda: Int8WeightOnlyConfig(version=2)
                )

    tf.fuse_lora()

    # Merge wrote into `base_layer.weight`, so the merged values survive
    # unloading. This restores plain nn.Linear modules and frees the adapter.
    tf.unload_lora()
    torch.cuda.empty_cache()

    # If requantization silently no-ops, the layer keeps its dequantized bf16
    # weight and the DiT doubles to ~62 GB. That is not an exception, so check
    # rather than trust.
    after = quantized_params()
    if after < before:
        raise RuntimeError(
            f"Turbo LoRA merge left {before - after} of {before} weights "
            "dequantized — torchao did not requantize them, so the transformer "
            "is now part BF16 and will not fit. Re-run this cell with "
            "FUSE_TURBO_LORA = False, or try Int8WeightOnlyConfig(version=1)."
        )

    tf._h3_turbo_lora_fused = float(strength)
    print(f"  ✓ merged — {after} weights still INT8")


if FUSE_TURBO_LORA:
    if not LOW_MEMORY_MODE:
        pipe.transformer.to("cuda")   # merge on-device; far faster than CPU
    fuse_turbo_lora(TURBO_LORA_STRENGTH)


# ── Placement strategy ────────────────────────────────────────
# The fast path keeps the transformer on GPU. The low path streams
# transformer/text-encoder groups from CPU memory and keeps VAEs off GPU.
if LOW_MEMORY_MODE:
    apply_group_offloading(
        pipe.transformer,
        offload_type="block_level",
        num_blocks_per_group=1,
        **memory_offload_kwargs(),
    )
    apply_group_offloading(
        pipe.vae,
        offload_type="leaf_level",
        **memory_offload_kwargs(),
    )
    apply_group_offloading(
        pipe.audio_vae,
        offload_type="leaf_level",
        **memory_offload_kwargs(),
    )
else:
    pipe.transformer.to("cuda")   # fully resident — no per-step PCIe streaming
    apply_group_offloading(
        pipe.text_encoder.model,
        offload_type="leaf_level",
        **_offload,
    )
    pipe.vae.to("cuda")
    pipe.audio_vae.to("cuda")

# ── Status ────────────────────────────────────────────────────
free_gb, total_gb = (v / 1e9 for v in torch.cuda.mem_get_info())
print("\n✓ Pipeline ready")
print(f"  VRAM : {free_gb:.1f} GB free / {total_gb:.1f} GB total")
print(f"  Mode : {'LOW (CPU group offload)' if LOW_MEMORY_MODE else 'FAST (GPU-resident transformer)'}")
print(f"  VAEs : {'CPU group-offloaded' if LOW_MEMORY_MODE else 'on GPU permanently'}")
print(f"  LoRA : {f'Turbo merged @ {TURBO_LORA_STRENGTH:.2f} — use 4-8 steps' if FUSE_TURBO_LORA else 'none'}")
print(f"  Alloc: {torch.cuda.memory_allocated() / 2**30:.1f} GiB allocated / "
      f"{torch.cuda.memory_reserved() / 2**30:.1f} GiB reserved")
print("\nRun Cell 3 to open the generation GUI.")

# ╔══════════════════════════════════════════════════════════════╗
# ║  OPTIONAL — Ref2VA (image / video / audio reference → video) ║
# ║  Lazy-loaded: only downloads + quantizes transformer_ref/    ║
# ║  the first time Ref2V mode is actually used from the GUI.    ║
# ╚══════════════════════════════════════════════════════════════╝
# MiniMax-H3 ships TWO transformer partitions in one repo:
#   transformer/      → t2va / fl2va   (loaded above, resident on GPU)
#   transformer_ref/  → ref2va         (omni-reference: up to 9 images,
#                                        3 videos, 3 audio clips — 12 max)
# Both partitions share every other component already loaded (VAEs,
# text encoder, schedulers), so ref2va only needs transformer_ref's
# own weights registered on `pipe` — nothing else re-downloads.
#
# This is NOT loaded eagerly: it's another ~31 GB quantized. In low-memory
# mode it is also kept in CPU memory and streamed on demand.

_ref2va_state = {"loaded": False}


def ensure_ref2va_loaded():
    """Idempotent — quantizes and registers transformer_ref on `pipe`
    the first time it's needed. Safe to call before every ref2va
    generation; subsequent calls are a no-op."""
    if _ref2va_state["loaded"]:
        return

    print("Loading Ref2VA transformer (INT8, first time only, ~31 GB) …")
    transformer_ref = MiniMaxH3Transformer3DModel.from_pretrained(
        REPO,
        subfolder="transformer_ref",
        dtype=torch.bfloat16,
        quantization_config=TorchAoConfig(
            Int8WeightOnlyConfig(version=2),
            modules_to_not_convert=TRANSFORMER_SKIP,
        ),
        low_cpu_mem_usage=True,
        device_map="cuda",          # quantize on GPU, not host RAM
    )
    transformer_ref.requires_grad_(False)
    transformer_ref = transformer_ref.to("cpu")   # hold in CPU memory for on-demand use
    torch.cuda.empty_cache()

    pipe.update_components(transformer_ref=transformer_ref)

    # Block-level CPU offload limits GPU peaks without disk serialization.
    apply_group_offloading(
        pipe.transformer_ref,
        offload_type="block_level",
        num_blocks_per_group=1,
        **memory_offload_kwargs(),
    )

    _ref2va_state["loaded"] = True
    print("  ✓ Ref2VA transformer ready (streamed from CPU memory on use)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Generation GUI                                    ║
# ║  Re-run this cell at any time to reset the interface.       ║
# ╚══════════════════════════════════════════════════════════════╝
import torch, time, os, io, math, traceback, subprocess, tempfile
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Video, HTML, clear_output
import PIL.Image

try:
    from diffusers.utils.export_utils import encode_video
except ImportError:
    from diffusers.utils import encode_video   # fallback import path

# Ref2VA reference dataclasses — used only by R2V mode below.
# `ensure_ref2va_loaded()` and `_ref2va_state` come from Cell 2.
from diffusers.modular_pipelines.minimax_h3 import (
    MiniMaxH3ImageReference,
    MiniMaxH3VideoReference,
    MiniMaxH3AudioReference,
)

OUTPUT_DIR = "/content/h3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Session state for "Extend from last" ─────────────────────
# "Extend from last" chains fl2va calls: it feeds the last frame of
# the previous clip back in as the new first-frame condition — the
# standard continuation trick for that workflow.
#
# True reference-guided generation (an existing video/image/audio as
# a *style/subject/motion reference*, not just a continuation anchor)
# is the separate ref2va workflow, added below as R2V mode. It uses
# a different transformer partition (transformer_ref/) that's loaded
# lazily on first use via ensure_ref2va_loaded() from Cell 2.
session_state = {
    "clips": [],
    "last_frame": None,

    # Resolution of the most recently generated clip.
    "last_width": None,
    "last_height": None,
}

# ══════════════════════════════════════════════════════════════
# ── Helpers ───────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════

def snap_frames(seconds: int) -> int:
    """H3 video VAE requires num_frames = 17n + 5 at 24 fps.
    Snap the requested duration to the nearest valid count.
    n=21 would give 362 frames, which exceeds the model's 360-frame
    (15.0s) ceiling — so the max valid n is 20 (345 frames, 14.375s)."""
    n = round((seconds * 24 - 5) / 17)
    n = max(0, min(20, n))          # clamps to 5 s – 14.375 s
    return 17 * n + 5

def actual_seconds(frames: int) -> str:
    s = frames / 24
    return f"{s:.2f}s"

def read_upload(w: widgets.FileUpload) -> PIL.Image.Image | None:
    """Return the first uploaded PIL image, or None."""
    if not w.value:
        return None
    raw = list(w.value.values())[0]["content"]
    return PIL.Image.open(io.BytesIO(raw)).convert("RGB")

def fmt_bytes(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

def extract_last_frame(frames) -> PIL.Image.Image:
    """Pull the final frame out of a generated clip's frame tensor/array
    as a PIL image, for use as the next clip's first-frame condition."""
    arr = frames.numpy() if hasattr(frames, "numpy") else np.array(frames)
    last = arr[-1]
    if last.dtype != np.uint8:
        last = (last * 255.0).clip(0, 255).astype(np.uint8) if last.max() <= 1.0 else last.astype(np.uint8)
    return PIL.Image.fromarray(last).convert("RGB")

def concat_clips(clip_paths, output_path):
    """Stitch clips into one file via ffmpeg's concat demuxer (stream
    copy, no re-encode — fast). Requires matching codec/resolution/fps
    across clips, which holds as long as resolution wasn't changed
    between generations in this session."""
    list_file = output_path + ".txt"
    with open(list_file, "w") as f:
        for p in clip_paths:
            f.write(f"file '{os.path.abspath(p)}'\n")
    cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_file, "-c", "copy", output_path]
    proc = subprocess.run(cmd, capture_output=True)
    os.remove(list_file)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.decode(errors="ignore")[-800:])
    return output_path

# Rough per-step estimates on A100 80 GB INT8.
# These are based on architecture scaling from community benchmarks —
# actual times will vary (±50 %) until published A100+Diffusers results exist.
STEP_SECONDS = {
    (864,  480): 4,
    (960,  544): 6,
    (1280, 704): 14,
    (1344, 768): 20,
    (480,  864): 4,
    (544,  960): 6,
    (640,  640): 7,
}
TEXT_ENC_SECONDS = 90   # Qwen3-VL-32B text encoding + offload

def rough_eta(width, height, steps, num_frames) -> str:
    """`steps` is the model-evaluation count, i.e. the slider value."""
    step_s  = STEP_SECONDS.get((width, height), 10)
    # VAE decode scales with frames (roughly linear)
    vae_s   = num_frames * 0.3
    total_s = TEXT_ENC_SECONDS + step_s * steps + vae_s
    mins, secs = divmod(int(total_s), 60)
    return f"~{mins}m {secs:02d}s"

# ══════════════════════════════════════════════════════════════
# ── Widget definitions ────────────────────────────────────────
# ══════════════════════════════════════════════════════════════

S  = {"description_width": "135px"}
LW = widgets.Layout(width="580px")
MW = widgets.Layout(width="380px")

# ── Mode ──────────────────────────────────────────────────────
w_mode = widgets.ToggleButtons(
    options=[
        "T2V — Text to Video",
        "I2V — Image to Video",
        "R2V — Reference to Video",
    ],
    value="T2V — Text to Video",
    description="Mode:",
    button_style="info",
    style=S,
)

# ── Prompt ────────────────────────────────────────────────────
w_prompt = widgets.Textarea(
    value=(
        "A golden retriever runs joyfully along a sun-drenched beach, "
        "waves crashing in the background, upbeat acoustic guitar, "
        "warm golden-hour light, slow motion, cinematic"
    ),
    description="Prompt:",
    rows=4, layout=LW, style=S,
)
w_prompt_hint = widgets.HTML(
    "<small style='color:#888'>"
    "No negative prompt or guidance scale — both are distilled into the weights. "
    "Describe audio, camera, and lighting in the prompt itself.</small>"
)

# ── Resolution ────────────────────────────────────────────────
# 960×544 is 2.3× faster per step than 1344×768 (official benchmark).
RESOLUTIONS = {
    "480p   864×480   ~4 s/step  ← fastest":  (864,  480),
    "540p   960×544   ~6 s/step  ← recommended": (960, 544),
    "720p  1280×704  ~14 s/step": (1280, 704),   # 720 isn't a multiple of 32 — snapped to 704
    "768p  1344×768  ~20 s/step  ← native max": (1344, 768),
    "Portrait  480×864  ~4 s/step": (480,  864),
    "Portrait  544×960  ~6 s/step": (544,  960),
    "Square    640×640  ~7 s/step": (640,  640),
    "Auto — match I2V input aspect ratio": None
}
w_res = widgets.Dropdown(
    options=list(RESOLUTIONS.keys()),
    value="Auto — match I2V input aspect ratio",
    description="Resolution:",
    layout=widgets.Layout(width="520px"),
    style=S,
)

AUTO_QUALITY = {
    "480p — fastest": (864, 480),
    "540p — recommended": (960, 544),
    "720p — high quality": (1280, 704),
    "768p — maximum": (1344, 768),
}
w_auto_quality = widgets.Dropdown(
    options=list(AUTO_QUALITY.keys()),
    value="540p — recommended",
    description="Auto quality:",
    layout=widgets.Layout(width="520px"),
    style=S,
)

def get_matching_resolution(
    image: PIL.Image.Image,
    base_width: int,
    base_height: int,
    min_dim: int = 32,
    max_dim: int = 1344,
):
    """
    Calculate an output resolution with approximately the same aspect
    ratio as the input image while maintaining roughly the same pixel
    budget as the selected resolution.

    Dimensions are snapped to multiples of 32 and clamped to
    [min_dim, max_dim] on each side (H3's own canvas tops out at a
    768px short edge / 1344px long edge on the released checkpoint).
    """
    src_w, src_h = image.size
    aspect = src_w / src_h

    # Pixel budget from selected resolution.
    target_pixels = base_width * base_height

    # Solve:
    #   width / height = aspect
    #   width * height ≈ target_pixels
    out_w = math.sqrt(target_pixels * aspect)
    out_h = out_w / aspect

    # Snap to multiples of 32, then clamp into range.
    out_w = round(out_w / 32) * 32
    out_h = round(out_h / 32) * 32
    out_w = max(min_dim, min(max_dim, out_w))
    out_h = max(min_dim, min(max_dim, out_h))

    return int(out_w), int(out_h)

# ── Duration ──────────────────────────────────────────────────
w_dur = widgets.IntSlider(
    value=5, min=5, max=15, step=1,
    description="Duration (s):",
    continuous_update=False, style=S, layout=LW,
)
w_dur_hint = widgets.HTML()

def update_dur_hint(*_):
    f = snap_frames(w_dur.value)

    resolution = RESOLUTIONS[w_res.value]

    if resolution is None:
        resolution_text = "Auto — matches I2V input aspect ratio"
        eta_text = "depends on input image aspect ratio"
    else:
        w, h = resolution
        resolution_text = f"{w}×{h}"
        eta_text = rough_eta(
            w,
            h,
            w_steps.value,
            f,
        )

    w_dur_hint.value = (
        f"<small style='color:#888'>"
        f"→ {f} frames ({actual_seconds(f)} actual at 24 fps, "
        f"snapped to nearest 17n+5 VAE boundary) &nbsp;·&nbsp; "
        f"{resolution_text} &nbsp;·&nbsp; "
        f"ETA with {w_steps.value} steps: {eta_text}"
        f"</small>"
    )
w_dur.observe(update_dur_hint, names="value")

def update_auto_quality_visibility(*_):
    is_auto = RESOLUTIONS[w_res.value] is None

    w_auto_quality.layout.display = (
        "" if is_auto else "none"
    )

w_res.observe(
    update_auto_quality_visibility,
    names="value",
)

update_auto_quality_visibility()

# ── Steps ─────────────────────────────────────────────────────
# This slider is the number of model evaluations — what "steps" means
# everywhere else (ComfyUI, the LoRA card, the reference generate.py).
# MiniMaxH3Scheduler counts the terminal sigma 0 as a grid point, so the
# pipeline is handed `steps + 1`; the conversion happens at the call site.
w_steps = widgets.IntSlider(
    value=9, min=4, max=25, step=1,
    description="Steps:",
    continuous_update=False, style=S, layout=LW,
)
w_steps.observe(update_dur_hint, names="value")
w_steps_hint = widgets.HTML(
    "<small style='color:#888'>"
    "Model evaluations. Below 6 quality degrades noticeably.</small>"
)

# ── Turbo LoRA (read-only status) ─────────────────────────────
# The merge happens in the model-loading cell, gated by FUSE_TURBO_LORA.
# There is nothing to toggle here: once merged, the adapter is part of the
# transformer's weights for the session.
def h3_turbo_lora_fused_strength():
    return getattr(pipe.transformer, "_h3_turbo_lora_fused", None)


def update_lora_controls(*_):
    fused = h3_turbo_lora_fused_strength()
    if fused is None:
        text = (
            "<b>Turbo LoRA</b>: not merged &nbsp;—&nbsp; "
            "<small style='color:#888'>set <code>FUSE_TURBO_LORA = True</code> "
            "in the model-loading cell and re-run it to enable the few-step "
            "adapter.</small>"
        )
    else:
        text = (
            f"<b style='color:green'>Turbo LoRA merged</b> @ {fused:.2f} "
            "&nbsp;—&nbsp; <small style='color:#888'>use 4-8 steps."
        )
        if "R2V" in w_mode.value:
            text += " R2V denoises with transformer_ref, which is unaffected."
        text += "</small>"
    w_lora_hint.value = text


w_lora_hint = widgets.HTML()
w_mode.observe(update_lora_controls, names="value")
update_lora_controls()

w_lora_box = widgets.VBox(
    [w_lora_hint],
    layout=widgets.Layout(
        border="1px dashed #bbb",
        padding="8px 10px",
        margin="6px 0",
        width="580px",
    ),
)


# ── Seed ──────────────────────────────────────────────────────
w_seed_rand = widgets.Checkbox(
    value=True, description="Random seed",
    indent=False, layout=widgets.Layout(width="160px"),
)
w_seed_val = widgets.IntText(
    value=42, description="Seed:",
    style=S, layout=widgets.Layout(width="210px", visibility="hidden"),
)
def on_seed_toggle(c):
    w_seed_val.layout.visibility = "hidden" if c["new"] else "visible"
w_seed_rand.observe(on_seed_toggle, names="value")

# ── I2V uploads ───────────────────────────────────────────────
w_first = widgets.FileUpload(
    accept="image/*", multiple=False,
    description="First frame *",
    layout=widgets.Layout(width="280px"), style=S,
)
w_last = widgets.FileUpload(
    accept="image/*", multiple=False,
    description="Last frame (opt.)",
    layout=widgets.Layout(width="280px"), style=S,
)
w_first_preview = widgets.Output(layout=widgets.Layout(width="260px"))
w_last_preview  = widgets.Output(layout=widgets.Layout(width="260px"))

def show_preview(upload_widget, preview_output):
    img = read_upload(upload_widget)
    if img is None:
        return
    preview_output.clear_output(wait=True)
    with preview_output:
        thumb = img.copy()
        thumb.thumbnail((200, 200))
        display(thumb)

w_first.observe(lambda _: show_preview(w_first, w_first_preview), names="value")
w_last.observe(lambda _: show_preview(w_last,  w_last_preview),  names="value")

w_i2v_box = widgets.VBox([
    widgets.HTML(
        "<b>Image-to-Video</b> &nbsp;—&nbsp; "
        "first frame is required; last frame is optional.<br>"
        "<small style='color:#888'>"
        "Canvas aspect ratio follows the first frame's dimensions. "
        "The Resolution setting above still controls pixel count.</small>"
    ),
    widgets.HBox([
        widgets.VBox([w_first, w_first_preview]),
        widgets.VBox([w_last,  w_last_preview]),
    ]),
], layout=widgets.Layout(
    display="none",
    border="1px dashed #bbb",
    padding="10px", margin="6px 0", border_radius="6px",
))

# ── R2V uploads (ref2va: image / video / audio references) ────
# ref2va conditions on an ORDERED list of references (labelled
# <Picture 1>, <Video 1>, <Audio 1>, ... in the prompt and on the
# shared rotary clock) rather than binding the output's geometry to
# any single input. Order here is: image ref, then video ref, then
# audio ref — reordering the same uploads is a different request.
w_ref_image = widgets.FileUpload(
    accept="image/*", multiple=False,
    description="Image ref (opt.)",
    layout=widgets.Layout(width="280px"), style=S,
)
w_ref_video = widgets.FileUpload(
    accept="video/*", multiple=False,
    description="Video ref *",
    layout=widgets.Layout(width="280px"), style=S,
)
w_ref_audio = widgets.FileUpload(
    accept="audio/*", multiple=False,
    description="Audio ref (opt.)",
    layout=widgets.Layout(width="280px"), style=S,
)
w_ref_image_preview = widgets.Output(layout=widgets.Layout(width="200px"))
w_ref_video_info     = widgets.HTML("<small style='color:#888'>no file</small>")
w_ref_audio_info     = widgets.HTML("<small style='color:#888'>no file</small>")

def show_ref_image_preview(_):
    img = read_upload(w_ref_image)
    w_ref_image_preview.clear_output(wait=True)
    if img is None:
        return
    with w_ref_image_preview:
        thumb = img.copy()
        thumb.thumbnail((160, 160))
        display(thumb)

def show_upload_info(w: widgets.FileUpload, info: widgets.HTML, label: str):
    if not w.value:
        info.value = "<small style='color:#888'>no file</small>"
        return
    meta = list(w.value.values())[0]
    size = fmt_bytes(meta["metadata"]["size"])
    name = meta["metadata"]["name"]
    info.value = f"<small style='color:#888'>{label}: {name} ({size})</small>"

w_ref_image.observe(show_ref_image_preview, names="value")
w_ref_video.observe(lambda _: show_upload_info(w_ref_video, w_ref_video_info, "video"), names="value")
w_ref_audio.observe(lambda _: show_upload_info(w_ref_audio, w_ref_audio_info, "audio"), names="value")

w_r2v_box = widgets.VBox([
    widgets.HTML(
        "<b>Reference-to-Video</b> (ref2va) &nbsp;—&nbsp; "
        "condition on an existing video (motion/camera/style), plus "
        "optional image (subject/style) and audio (voice/music) "
        "references — up to 9 images, 3 videos, 3 audio, 12 total.<br>"
        "<small style='color:#888'>"
        "This does NOT bind to the reference's resolution — the "
        "canvas below defaults to 16:9 unless you pick a resolution. "
        "First use downloads and quantizes an extra ~31 GB transformer "
        "partition (transformer_ref/), then it's cached for the "
        "session.</small>"
    ),
    widgets.HBox([
        widgets.VBox([w_ref_image, w_ref_image_preview]),
        widgets.VBox([w_ref_video, w_ref_video_info]),
        widgets.VBox([w_ref_audio, w_ref_audio_info]),
    ]),
], layout=widgets.Layout(
    display="none",
    border="1px dashed #bbb",
    padding="10px", margin="6px 0", border_radius="6px",
))

def on_mode(c):
    w_i2v_box.layout.display = "" if "I2V" in c["new"] else "none"
    w_r2v_box.layout.display = "" if "R2V" in c["new"] else "none"
w_mode.observe(on_mode, names="value")

# ── Buttons & status ──────────────────────────────────────────
w_btn_gen = widgets.Button(
    description="▶  Generate",
    button_style="success",
    layout=widgets.Layout(width="150px", height="38px"),
)
w_btn_extend = widgets.Button(
    description="⏩ Extend from last",
    button_style="primary",
    disabled=True,   # enabled once a clip exists this session
    layout=widgets.Layout(width="170px", height="38px"),
)
w_extend_preview = widgets.Output(layout=widgets.Layout(width="80px", height="45px"))
w_btn_clear = widgets.Button(
    description="Clear output",
    button_style="warning",
    layout=widgets.Layout(width="130px", height="38px"),
)
w_status  = widgets.HTML("<span style='color:#555'>Ready — click Generate</span>")
w_out     = widgets.Output()

# ── Progress bar (shown while generating) ─────────────────────
w_progress = widgets.FloatProgress(
    value=0, min=0, max=1,
    bar_style="info",
    layout=widgets.Layout(width="560px", visibility="hidden"),
)

# ══════════════════════════════════════════════════════════════
# ── Generation callback ───────────────────────────────────────
# ══════════════════════════════════════════════════════════════

def _do_generation(is_extend: bool):
    w_btn_gen.disabled      = True
    w_btn_extend.disabled   = True
    w_btn_gen.description   = "⏳ Running…"
    w_progress.layout.visibility = "visible"
    w_progress.value = 0.05

    try:
        # ── Extend guard ────────────────────────────────────────
        if is_extend and session_state["last_frame"] is None:
            raise ValueError(
                "No previous clip in this session yet — "
                "click Generate at least once before Extending."
            )

        # ── Gather parameters ─────────────────────────────────
        num_frames = snap_frames(w_dur.value)
        steps = w_steps.value
        is_r2v = False if is_extend else ("R2V" in w_mode.value)
        is_i2v = True if is_extend else ("I2V" in w_mode.value)

        # ── Determine output resolution ───────────────────────────

        resolution = RESOLUTIONS[w_res.value]

        if is_extend:
            # Extensions MUST use the exact same resolution as the
            # previous clip so that concatenation remains compatible.
            if (
                session_state["last_width"] is None
                or session_state["last_height"] is None
            ):
                raise ValueError(
                    "Previous clip resolution is unavailable."
                )

            width = session_state["last_width"]
            height = session_state["last_height"]

        elif is_r2v:
            # ref2va doesn't bind the output canvas to any reference's
            # own resolution — references are encoded at their own
            # size regardless of what the generated video's canvas is.
            # There's no first-frame image to match an aspect ratio
            # against here, so "Auto" just falls back to the chosen
            # Auto-quality preset (H3's own default is ~16:9 anyway).
            if resolution is None:
                width, height = AUTO_QUALITY[w_auto_quality.value]
            else:
                width, height = resolution

        elif resolution is None:
          if is_i2v:
              first_img = read_upload(w_first)

              if first_img is None:
                  raise ValueError(
                      "Auto resolution requires a first-frame image."
                  )

              base_width, base_height = AUTO_QUALITY[
                  w_auto_quality.value
              ]

              width, height = get_matching_resolution(
                  first_img,
                  base_width=base_width,
                  base_height=base_height,
                  min_dim=320,
                  max_dim=1344,
              )
          else:
              # T2V has no input image to match an aspect ratio against,
              # so "Auto" just falls back to the chosen Auto-quality
              # preset (default: 540p) — same fallback ref2va already uses.
              width, height = AUTO_QUALITY[w_auto_quality.value]

        else:
            width, height = resolution
        seed          = (
            int(time.time() * 1000) % (2**31 - 1)
            if w_seed_rand.value else w_seed_val.value
        )
        generator = torch.Generator().manual_seed(seed)

        eta = rough_eta(width, height, steps, num_frames)
        mode_label = "Extending" if is_extend else "Encoding prompt"
        w_status.value = (
            f"<span style='color:#555'>"
            f"⏳ {mode_label} … "
            f"({num_frames} frames @ {width}×{height}, "
            f"{steps} steps, ETA {eta})</span>"
        )

        # ── Build call kwargs ─────────────────────────────────
        call_kwargs = dict(
            prompt              = w_prompt.value,
            num_frames          = num_frames,
            width               = width,
            height              = height,
            # +1: the scheduler counts the terminal sigma 0 as a grid point.
            num_inference_steps = steps + 1,
            generator           = generator,
            output              = ["videos", "audio", "sampling_rate"],
        )

        if is_extend:
            # Condition on the last frame of the previous clip instead
            # of a manual upload — this is the continuation mechanism.
            call_kwargs["image"] = session_state["last_frame"]
        elif is_r2v:
            # ── Gather ordered references for ref2va ────────────
            references = []

            ref_img = read_upload(w_ref_image)
            if ref_img is not None:
                references.append(MiniMaxH3ImageReference(image=ref_img))

            if not w_ref_video.value:
                raise ValueError(
                    "R2V mode requires a reference video — "
                    "please upload one before generating."
                )
            # `from_file` decodes through PyAV, which carries the real
            # frame rate (and soundtrack) along. Building this from
            # `load_video()` frames instead would silently drop the
            # frame rate and condition at the wrong speed.
            video_bytes = list(w_ref_video.value.values())[0]["content"]
            with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tf:
                tf.write(bytes(video_bytes))
                video_tmp_path = tf.name
            try:
                references.append(MiniMaxH3VideoReference.from_file(video_tmp_path))
            finally:
                os.remove(video_tmp_path)

            if w_ref_audio.value:
                audio_bytes = list(w_ref_audio.value.values())[0]["content"]
                suffix = os.path.splitext(
                    list(w_ref_audio.value.values())[0]["metadata"]["name"]
                )[1] or ".wav"
                with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tf:
                    tf.write(bytes(audio_bytes))
                    audio_tmp_path = tf.name
                try:
                    references.append(MiniMaxH3AudioReference.from_file(audio_tmp_path))
                finally:
                    os.remove(audio_tmp_path)

            call_kwargs["references"] = references

            # Lazy-load the transformer_ref partition (once per session).
            w_status.value = (
                "<span style='color:#555'>"
                "⏳ Preparing Ref2VA transformer "
                "(first use only, can take ~15 min) …</span>"
            )
            ensure_ref2va_loaded()

        elif is_i2v:
          first_img = read_upload(w_first)

          if first_img is None:
              raise ValueError(
                  "I2V mode requires a first frame — "
                  "please upload an image before generating."
              )

          # Resize proportionally to the exact generation dimensions.
          # This does NOT distort the image because width/height already
          # match its aspect ratio in Auto mode.
          first_img = first_img.resize(
              (width, height),
              PIL.Image.Resampling.LANCZOS,
          )

          call_kwargs["image"] = first_img

          last_img = read_upload(w_last)

          if last_img is not None:
              # Same treatment for optional last frame.
              last_img = last_img.resize(
                  (width, height),
                  PIL.Image.Resampling.LANCZOS,
              )

              call_kwargs["last_image"] = last_img

        # ── Turbo LoRA ────────────────────────────────────────
        # Already merged into the weights if fused; nothing to apply per run.
        # R2V denoises with transformer_ref, which never gets the merge.
        lora_strength = h3_turbo_lora_fused_strength()
        lora_on       = lora_strength is not None and not is_r2v

        w_progress.value = 0.1
        w_status.value = (
            f"<span style='color:#555'>"
            f"⏳ Denoising{' (Turbo LoRA)' if lora_on else ''} … "
            f"({num_frames} frames @ {width}×{height}, "
            f"{steps} steps, seed {seed}, ETA {eta})</span>"
        )

        # ── Run ───────────────────────────────────────────────
        t0      = time.time()
        results = pipe(**call_kwargs)
        elapsed = time.time() - t0

        w_progress.value = 0.90

        # ── Save to MP4 with audio ────────────────────────────
        ts       = int(time.time())
        tag      = "ext" if is_extend else ("ref" if is_r2v else "gen")
        filename = f"h3_{ts}_{tag}_s{seed}.mp4"
        out_path = os.path.join(OUTPUT_DIR, filename)

        try:
            encode_video(
                results["videos"][0],
                fps=24,
                output_path=out_path,
                audio=results["audio"][0],
                audio_sample_rate=results["sampling_rate"],
            )
        except Exception as enc_err:
            # Fallback: save frames as a silent GIF if encode_video fails
            print(f"  encode_video failed ({enc_err}), saving frames as GIF …")
            gif_path = out_path.replace(".mp4", ".gif")
            frames   = results["videos"][0]
            if hasattr(frames, "numpy"):
                frames_list = [
                    PIL.Image.fromarray(f) for f in frames.numpy()
                ]
            else:
                frames_list = frames
            frames_list[0].save(
                gif_path, save_all=True,
                append_images=frames_list[1:], loop=0, duration=42,
            )
            out_path = gif_path

        # ── Update session state for future extends ────────────
        if out_path.endswith(".mp4"):
            session_state["clips"].append(out_path)
            session_state["last_frame"] = extract_last_frame(results["videos"][0])
            # Remember the exact resolution for future extensions.
            session_state["last_width"] = width
            session_state["last_height"] = height

            w_btn_extend.disabled = False

            w_extend_preview.clear_output(wait=True)

            with w_extend_preview:
                thumb = session_state["last_frame"].copy()
                thumb.thumbnail((70, 70))
                display(thumb)
            w_btn_extend.disabled = False
            w_extend_preview.clear_output(wait=True)
            with w_extend_preview:
                thumb = session_state["last_frame"].copy()
                thumb.thumbnail((70, 70))
                display(thumb)

        w_progress.value = 1.0

        # ── Display ───────────────────────────────────────────
        size_str = fmt_bytes(os.path.getsize(out_path))
        mins, secs = divmod(int(elapsed), 60)
        elapsed_str = f"{mins}m {secs:02d}s" if mins else f"{secs}s"
        clip_n = len(session_state["clips"])

        w_status.value = (
            f"<b style='color:green'>✓ Done in {elapsed_str}</b> — "
            f"{width}×{height} · {actual_seconds(num_frames)} · "
            f"{steps} steps · seed {seed} · {size_str}"
            + (f" · clip {clip_n} in session" if is_extend else "")
        )

        vram_free = torch.cuda.mem_get_info()[0] / 1e9
        with w_out:
            display(HTML(
                f"<div style='font-family:monospace;font-size:11px;"
                f"color:#888;margin:6px 0;'>"
                f"{'Extend' if is_extend else w_mode.value.split('—')[0].strip()} &nbsp;·&nbsp; "
                f"{width}×{height} &nbsp;·&nbsp; "
                f"{num_frames} frames &nbsp;·&nbsp; "
                f"{steps} steps &nbsp;·&nbsp; "
                + (f"turbo LoRA @ {lora_strength:.2f} &nbsp;·&nbsp; "
                   if lora_on else "")
                + f"seed {seed} &nbsp;·&nbsp; "
                f"wall-clock {elapsed_str} &nbsp;·&nbsp; "
                f"VRAM free {vram_free:.1f} GB"
                f"</div>"
            ))
            if out_path.endswith(".mp4"):
                display(Video(
                    out_path, embed=True,
                    width=min(width, 640),
                    height=min(height, 360),
                ))
            else:
                img_out = PIL.Image.open(out_path)
                display(img_out)
            print(f"Saved: {out_path}  ({size_str})")

            # ── Stitch only the previous clip + the newly generated clip ──
            if is_extend and len(session_state["clips"]) >= 2:
                previous_clip = session_state["clips"][-2]
                current_clip  = session_state["clips"][-1]

                combined_path = os.path.join(
                    OUTPUT_DIR,
                    f"session_{len(session_state['clips'])}_clips.mp4"
                )

                try:
                    concat_clips(
                        [previous_clip, current_clip],
                        combined_path
                    )

                    combined_size = fmt_bytes(os.path.getsize(combined_path))

                    print(
                        f"\nPrevious clip + new extension stitched "
                        f"({combined_size}):"
                    )

                    display(Video(
                        combined_path,
                        embed=True,
                        width=min(width, 640),
                        height=min(height, 360),
                    ))

                except Exception as concat_err:
                    print(
                        "\nCouldn't stitch previous clip + new extension "
                        "(keep resolution/fps constant across extends):"
                    )
                    print(str(concat_err)[:400])

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        w_status.value = (
            "<span style='color:red'>✗ CUDA out of memory. "
            "Try a lower resolution or shorter duration.</span>"
        )
        with w_out:
            print("OOM — suggestions:")
            print("  • Lower resolution (try 864×480)")
            print("  • Shorter duration (try 5 s)")
            print("  • Fewer steps (try 6)")

    except ValueError as e:
        w_status.value = f"<span style='color:red'>✗ {e}</span>"

    except Exception as e:
        w_status.value = f"<span style='color:red'>✗ {type(e).__name__}: {e}</span>"
        with w_out:
            traceback.print_exc()

    finally:
        w_btn_gen.disabled              = False
        w_btn_extend.disabled           = session_state["last_frame"] is None
        w_btn_gen.description           = "▶  Generate"
        w_progress.layout.visibility    = "hidden"

def on_generate(_):
    _do_generation(is_extend=False)

def on_extend(_):
    _do_generation(is_extend=True)

def on_clear(_):
    w_out.clear_output()

    w_status.value = (
        "<span style='color:#555'>"
        "Ready — click Generate"
        "</span>"
    )

    session_state["clips"] = []
    session_state["last_frame"] = None
    session_state["last_width"] = None
    session_state["last_height"] = None

    w_btn_extend.disabled = True
    w_extend_preview.clear_output()

w_btn_gen.on_click(on_generate)
w_btn_extend.on_click(on_extend)
w_btn_clear.on_click(on_clear)

# Trigger initial hint
update_dur_hint()

# ══════════════════════════════════════════════════════════════
# ── Assemble and display ──────────────────────────────────────
# ══════════════════════════════════════════════════════════════

HR = widgets.HTML(
    "<hr style='border:0;border-top:1px solid #e0e0e0;margin:8px 0'>"
)

panel = widgets.VBox([
    widgets.HTML(
        "<h3 style='margin:4px 0;font-family:sans-serif'>"
        "🎬 MiniMax H3 Generator</h3>"
        "<div style='font-size:12px;color:#888;margin-bottom:10px'>"
        "Diffusers · INT8 · A100 80 GB · joint video + stereo audio</div>"
    ),
    w_mode,
    w_prompt,
    w_prompt_hint,
    w_i2v_box,
    w_r2v_box,
    HR,
    w_res,
    w_auto_quality,
    w_dur,
    w_dur_hint,
    w_steps,
    w_steps_hint,
    w_lora_box,
    widgets.HBox([w_seed_rand, w_seed_val]),
    HR,
    widgets.HBox(
        [w_btn_gen, w_btn_extend, w_extend_preview, w_btn_clear],
        layout=widgets.Layout(gap="8px", align_items="center"),
    ),
    w_progress,
    w_status,
    w_out,
], layout=widgets.Layout(
    padding="16px",
    border="1px solid #d0d0d0",
    border_radius="8px",
    max_width="640px",
))

display(panel)

In [ ]:
print("Keep Alive")